# Model Training (Base Models) - M5 Forecasting Accuracy

Notebook ini mengimplementasikan **Tahap 6-7: Strategi Grouped Modeling & Model Training (Base Models)**
sesuai `Bab 3 Metode Penelitian` (subbab 3.2.6 dan 3.2.7) pada dokumen Pre-Thesis.

## Ringkasan metodologi yang diikuti

**Strategi Grouped Modeling (Tabel 3.3)** - tiga skema pengelompokan dibentuk dari struktur hierarkis M5:

| Strategi | Jumlah Kelompok | Kunci Grup |
|---|---|---|
| By Store | 10 | `store_id` |
| By Store + Category | 30 | `store_id`, `cat_id` |
| By Store + Department | 70 | `store_id`, `dept_id` |

**Model dasar (Bab 2 & Tabel 3.4-3.7)**: LightGBM (Tweedie), XGBoost (Tweedie), CatBoost (Tweedie) - ketiganya
dilatih **terpisah per kelompok** sesuai tiga strategi grouped modeling di atas (subbab 3.2.7, paragraf pembuka).
LSTM dilatih **secara global** pada seluruh 30.490 deret waktu (subbab 3.2.7.3c), bukan per kelompok, dengan
identitas hierarkis dimasukkan sebagai fitur kontekstual.

**Strategi forecasting multi-horizon (subbab 3.2.8)**:
- *Recursive* - satu model memprediksi seluruh horizon 28 hari secara iteratif; hasil prediksi hari
  sebelumnya dipakai sebagai input lag untuk hari berikutnya.
- *Direct/non-recursive* - satu model terpisah dilatih untuk setiap horizon h = 1..28, memprediksi h hari
  ke depan langsung dari fitur pada hari cutoff, tanpa memakai hasil prediksi sebagai input.

**Tabel 3.8 (Model Dasar Kandidat Ensemble)** mendaftar 20 baris konfigurasi (algoritma x strategi grouping x
strategi forecasting). Setiap baris **bukan satu model tunggal** - untuk LightGBM/XGBoost/CatBoost, satu baris
berarti satu model per kelompok pada strategi grouping tersebut (mis. "LightGBM By Store Recursive" = 10 model
individual, satu per toko). Untuk direct forecasting, dikalikan lagi dengan 28 horizon. Ringkasannya:

- LightGBM/XGBoost/CatBoost x (byStore + byStoreCategory + byStoreDepartment) x (recursive + direct)
  = 3 model x 110 kelompok x 2 strategi = artefak individual yang sangat banyak (recursive: 1 model/kelompok;
  direct: 28 model/kelompok).
- LSTM (global) x (recursive + direct) = 2 baris, dengan direct diimplementasikan sebagai 28 model terpisah
  (satu per horizon) mengikuti definisi direct forecasting pada subbab 3.2.8.2.

## Ruang lingkup notebook ini
- Pelatihan dilakukan pada **main split** (`train`: d_1-d_1913, `test_public`: d_1914-d_1941) hasil notebook
  01-03, **bukan** pada 4-fold time-series CV. Evaluasi CV, best-model-selection (Tahap 8-9), dan WRMSSE
  hierarkis penuh (12 level) didelegasikan ke notebook evaluasi berikutnya - melatih 3 model x 110 kelompok x
  2 strategi x 4 fold CV pada mesin dengan 10 GB RAM usable dan GPU 4 GB VRAM tidak realistis dalam satu
  notebook. Notebook ini fokus menghasilkan seluruh model dasar + prediksi 28 hari + metrik pada main split,
  sebagai fondasi untuk Tahap 8 (Model Evaluation) dan Tahap 10 (Ensemble Learning).
- Sumber data adalah output **notebook 03** (`train_features_full_by_store` / `test_features_full_by_store`,
  parquet per store) - bukan file `.csv.gz` gabungan - supaya tiap store dapat dimuat dan dibuang dari memory
  satu per satu (peak memory rendah, konsisten dengan strategi notebook 03).
- `MODE = "smoke"` memvalidasi seluruh pipeline (semua kombinasi grouping x forecasting x model) dengan subset
  kecil data dan hyperparameter ringan dalam hitungan menit, sebelum `MODE = "full"` dijalankan dengan
  hyperparameter resmi (Tabel 3.4-3.7) pada seluruh data. **Jalankan smoke dulu untuk memvalidasi, baru ganti
  ke full.**
- LSTM dilatih secara global sesuai metodologi, namun karena keterbatasan hardware (10 GB RAM usable, 4 GB
  VRAM), jumlah deret waktu yang dipakai untuk membangun sekuens LSTM dibatasi lewat parameter
  `LSTM_MAX_SERIES` (lihat sel konfigurasi) - ini adalah adaptasi praktis yang didokumentasikan secara
  eksplisit, bukan penyimpangan diam-diam dari metodologi.

## Output notebook
Untuk setiap kombinasi (strategi grouping, kelompok, strategi forecasting, model dasar):
- Model terlatih (`models/<config>/<group>.pkl` atau `.keras`)
- Prediksi 28 hari pada test set (`results/predictions/<config>__<group>.csv`)
- Metrik RMSE, MAE, MAPE, dan proxy WRMSSE per-kelompok (`results/base_model_manifest.csv`)

File `base_model_manifest.csv` berfungsi sebagai checkpoint - menjalankan ulang notebook akan melewati
kombinasi yang sudah selesai dilatih (idempotent), sehingga notebook dapat dihentikan dan dilanjutkan.

> **Catatan V6 (perbaikan bug metrik):** fungsi `group_wrmsse_proxy` diperbaiki supaya tidak lagi menghasilkan bobot mentah/negatif ketika total dollar_sales suatu kelompok <= 0 (penyebab WRMSSE_proxy LSTM sebelumnya keluar sebagai -714 / -1004). Lihat sel definisi metrik untuk detail. Manifest lama (`base_model_manifest.csv`) berisi baris yang dihitung dengan rumus LAMA - hapus atau hitung ulang baris tersebut sebelum dipakai di tahap evaluasi/ensemble.\n\n> **Catatan V7 (CV diterapkan):** training base model sekarang memakai **4-fold time-series CV expanding window (CV0-CV3)** dari `dataset/splits/cv_folds.csv` (notebook 01), bukan 1 main split tunggal - setiap kombinasi strategi grouping x model x forecasting dilatih ulang di keempat fold. Baris manifest lama (main split d_1-d_1913 vs d_1914-d_1941) tetap tersimpan berlabel `fold=MAIN` dan tidak terhapus/tertimpa. Perhatian: ini melipatgandakan waktu training kira-kira 4x dibanding sebelumnya.

## 1. Import Library

In [1]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import os
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from sklearn.metrics import mean_squared_error, mean_absolute_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("lightgbm/xgboost/catboost/tensorflow loaded.")
print("GPUs terdeteksi untuk TensorFlow:", gpus if gpus else "tidak ada (CPU only)")


lightgbm/xgboost/catboost/tensorflow loaded.
GPUs terdeteksi untuk TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

FORECAST_HORIZON = 28

# "smoke" -> validasi pipeline cepat dengan data & hyperparameter kecil.
# "full"  -> hyperparameter resmi (Tabel 3.4-3.7) pada seluruh data.
MODE = "full"

# ------------------------------------------------------------------
# Strategi grouped modeling (Tabel 3.3). Kunci grup memakai kolom
# string asli (bukan _le) karena keduanya tersedia di parquet hasil
# notebook 03, dan kolom string lebih mudah dipakai untuk filtering.
# ------------------------------------------------------------------
GROUPING_STRATEGIES = {
    "byStore": ["store_id"],
    "byStoreCategory": ["store_id", "cat_id"],
    "byStoreDepartment": ["store_id", "dept_id"],
}

FORECASTING_STRATEGIES = ["recursive", "direct"]

# LightGBM/XGBoost/CatBoost dilatih per kelompok (grouped).
TREE_BASE_MODELS = ["lightgbm", "xgboost", "catboost"]
# LSTM dilatih global (subbab 3.2.7.3c) - ditangani terpisah di bagian akhir notebook.
BASE_MODELS = TREE_BASE_MODELS + ["lstm"]

# Lag/rolling window yang bisa "bocor" nilai aktual dari dalam horizon 28 hari
# apabila dipakai apa adanya dari parquet saat recursive forecasting (parquet
# dihitung dari sales aktual, termasuk untuk hari-hari di test set). Untuk lag
# dan rolling window ini, recursive forecasting menghitung ulang secara dinamis
# memakai riwayat aktual + prediksi sebelumnya. Window >= 28 selalu merujuk
# hari sebelum cutoff sehingga aman dipakai langsung dari parquet.
SHORT_LAGS = [7, 14, 21]
SHORT_ROLLING_WINDOWS = [7, 14, 28]

EVAL_HOLD_DAYS = 28  # dipakai sebagai validation set untuk early stopping tree model

# ------------------------------------------------------------------
# Hyperparameter resmi (Tabel 3.4 - 3.6) untuk MODE="full".
# ------------------------------------------------------------------
LIGHTGBM_PARAMS_FULL = dict(
    objective="tweedie",
    tweedie_variance_power=1.1,
    learning_rate=0.03,
    n_estimators=1500,
    num_leaves=2047,
    min_data_in_leaf=4095,
    feature_fraction=0.5,
    bagging_fraction=0.5,
    bagging_freq=1,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

XGBOOST_PARAMS_FULL = dict(
    objective="reg:tweedie",
    tweedie_variance_power=1.1,
    learning_rate=0.03,
    n_estimators=1500,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

CATBOOST_PARAMS_FULL = dict(
    loss_function="Tweedie:variance_power=1.1",
    learning_rate=0.03,
    iterations=1500,
    depth=8,
    l2_leaf_reg=3.0,
    bagging_temperature=0.5,
    random_seed=SEED,
    verbose=False,
)

EARLY_STOPPING_ROUNDS_FULL = 100

LSTM_PARAMS_FULL = dict(
    window_size=28,
    lstm_units_1=128,
    lstm_units_2=64,
    dropout_rate=0.2,
    dense_units_1=32,
    dense_units_2=16,
    learning_rate=0.001,
    batch_size=256,
    epochs=100,
    early_stopping_patience=10,
)

# ------------------------------------------------------------------
# RAM-SAFETY UNTUK LSTM (laptop: ~9 GB RAM usable / 4 GB VRAM)
# ------------------------------------------------------------------
# LSTM_SEQ_STRIDE: bukan setiap hari dijadikan training sample - melompat
# `stride` hari antar window training. Ini yang paling menentukan RAM,
# karena tanpanya build_lstm_sequences membuat satu sample per (id, hari),
# yang untuk ribuan deret waktu x ~1800 hari bisa mencapai puluhan juta
# baris. stride=7 memangkas jumlah sample ~7x dengan overlap window yang
# masih cukup untuk LSTM belajar pola mingguan/bulanan.
LSTM_SEQ_STRIDE = 7

# Batas keras jumlah training sample (setelah stride) yang boleh dipakai
# sekali fit; kalau masih lebih banyak, disubsample acak lagi. Ini jaring
# pengaman kedua kalau LSTM_MAX_SERIES x panjang histori masih terlalu besar.
LSTM_MAX_TRAIN_SAMPLES = 250_000

# ------------------------------------------------------------------
# Versi ringan untuk MODE="smoke" - tujuannya memvalidasi bahwa SETIAP
# kombinasi (grouping x forecasting x model) berjalan tanpa error dan
# menghasilkan artefak dengan struktur yang benar, bukan mengejar akurasi.
# ------------------------------------------------------------------
LIGHTGBM_PARAMS_SMOKE = dict(
    objective="tweedie", tweedie_variance_power=1.1, learning_rate=0.1,
    n_estimators=40, num_leaves=31, min_data_in_leaf=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    random_state=SEED, n_jobs=-1, verbose=-1,
)
XGBOOST_PARAMS_SMOKE = dict(
    objective="reg:tweedie", tweedie_variance_power=1.1, learning_rate=0.1,
    n_estimators=40, max_depth=4, subsample=0.9, colsample_bytree=0.9,
    reg_alpha=0.1, reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=0,
)
CATBOOST_PARAMS_SMOKE = dict(
    loss_function="Tweedie:variance_power=1.1", learning_rate=0.1,
    iterations=40, depth=4, l2_leaf_reg=3.0, bagging_temperature=0.5,
    random_seed=SEED, verbose=False,
)
EARLY_STOPPING_ROUNDS_SMOKE = 10
LSTM_PARAMS_SMOKE = dict(
    window_size=28, lstm_units_1=16, lstm_units_2=8, dropout_rate=0.2,
    dense_units_1=8, dense_units_2=4, learning_rate=0.005, batch_size=64,
    epochs=5, early_stopping_patience=3,
)

if MODE == "smoke":
    LIGHTGBM_PARAMS = LIGHTGBM_PARAMS_SMOKE
    XGBOOST_PARAMS = XGBOOST_PARAMS_SMOKE
    CATBOOST_PARAMS = CATBOOST_PARAMS_SMOKE
    EARLY_STOPPING_ROUNDS = EARLY_STOPPING_ROUNDS_SMOKE
    LSTM_PARAMS = LSTM_PARAMS_SMOKE
    # subset kecil supaya smoke test selesai dalam hitungan menit, tapi tetap
    # menyentuh ketiga strategi grouping (>=2 store supaya byStore benar2 >1 grup)
    SMOKE_STORES = ["CA_1", "TX_1"]
    SMOKE_MAX_ITEMS_PER_STORE = 60
    LSTM_MAX_SERIES = 40
else:
    LIGHTGBM_PARAMS = LIGHTGBM_PARAMS_FULL
    XGBOOST_PARAMS = XGBOOST_PARAMS_FULL
    CATBOOST_PARAMS = CATBOOST_PARAMS_FULL
    EARLY_STOPPING_ROUNDS = EARLY_STOPPING_ROUNDS_FULL
    LSTM_PARAMS = LSTM_PARAMS_FULL
    SMOKE_STORES = None
    SMOKE_MAX_ITEMS_PER_STORE = None
    # Melatih LSTM global pada seluruh 30.490 deret waktu sekaligus (sequence
    # tensor window=28 x ~90 fitur) tidak muat pada 10 GB RAM usable / 4 GB VRAM.
    # LSTM_MAX_SERIES membatasi jumlah deret yang disampling untuk membangun
    # sekuens LSTM full-mode - adaptasi praktis terhadap keterbatasan hardware,
    # didokumentasikan di sini secara eksplisit. Naikkan nilai ini jika hardware
    # yang dipakai memiliki RAM/VRAM lebih besar.
    # Diturunkan dari 3000 - dikombinasikan dengan LSTM_SEQ_STRIDE dan
    # LSTM_MAX_TRAIN_SAMPLES, 1500 deret sudah representatif untuk versi
    # full pada laptop 9 GB RAM / 4 GB VRAM. Naikkan lagi jika hardware
    # yang dipakai punya RAM/VRAM lebih besar.
    LSTM_MAX_SERIES = 1500

print("MODE:", MODE)
print("GROUPING_STRATEGIES:", list(GROUPING_STRATEGIES.keys()))
print("FORECASTING_STRATEGIES:", FORECASTING_STRATEGIES)
print("BASE_MODELS:", BASE_MODELS)

MODE: full
GROUPING_STRATEGIES: ['byStore', 'byStoreCategory', 'byStoreDepartment']
FORECASTING_STRATEGIES: ['recursive', 'direct']
BASE_MODELS: ['lightgbm', 'xgboost', 'catboost', 'lstm']


## 2b. Deteksi GPU & Pengaturan Kecepatan

**Update setelah run pertama**: paralelisasi GPU per-thread ternyata TIDAK aman -
CatBoost melempar `device already requested 0` / `State == nullptr` saat dua
thread meminta device CUDA yang sama secara bersamaan, dan itu sampai
meng-crash kernel (bukan sekadar exception Python yang bisa ditangkap). Ada juga
OOM terpisah: `make_direct_training_frame` menyalin seluruh frame grup per
horizon, dan menjalankan beberapa salinan itu bersamaan untuk grup besar
(`byStore`, ratusan ribu-jutaan baris) melebihi RAM yang tersedia.

Perbaikan di versi ini:

1. **Worker GPU dipaksa 1 (sequential)** untuk model yang pakai GPU (CatBoost/XGBoost)
   - konteks CUDA dalam satu proses terbukti tidak aman dipakai beberapa thread
   sekaligus. Percepatan dari GPU itu sendiri (per-fit) tetap didapat; yang
   dihilangkan hanya overlap antar-horizon untuk model GPU.
2. **Worker CPU (LightGBM, atau model manapun saat GPU tidak dipakai) sekarang
   dihitung dinamis berdasarkan ukuran grup** - grup besar (`byStore`) otomatis
   dapat sedikit worker paralel (bisa turun ke 1), grup kecil (`byStoreDepartment`)
   tetap dapat banyak worker paralel, supaya total memory salinan frame yang
   berjalan bersamaan tidak pernah melebihi anggaran (`MEMORY_BUDGET_BYTES_DIRECT`).
3. **`append_manifest_row` sekarang menulis atomik** (tulis ke file sementara lalu
   `os.replace`), supaya kalau kernel crash lagi di tengah proses training, file
   manifest tidak pernah berakhir dalam keadaan setengah tertulis/corrupt.


In [3]:
GENERAL_GPU_AVAILABLE = bool(tf.config.list_physical_devices("GPU"))


def _detect_lightgbm_gpu() -> bool:
    """Probe fit kecil untuk cek apakah wheel lightgbm yang terpasang dikompilasi
    dengan dukungan GPU. Wheel pip default biasanya CPU-only, jadi ini sering
    mengembalikan False walau GPU tersedia untuk library lain - itu wajar, bukan bug."""
    try:
        probe_X = np.random.rand(64, 4).astype("float32")
        probe_y = np.random.rand(64).astype("float32")
        LGBMRegressor(device="gpu", n_estimators=2, verbose=-1).fit(probe_X, probe_y)
        return True
    except Exception:
        return False


LIGHTGBM_GPU_AVAILABLE = _detect_lightgbm_gpu() if GENERAL_GPU_AVAILABLE else False
XGBOOST_GPU_AVAILABLE = GENERAL_GPU_AVAILABLE
CATBOOST_GPU_AVAILABLE = GENERAL_GPU_AVAILABLE

if LIGHTGBM_GPU_AVAILABLE:
    LIGHTGBM_PARAMS["device"] = "gpu"
if XGBOOST_GPU_AVAILABLE:
    XGBOOST_PARAMS["device"] = "cuda"
if CATBOOST_GPU_AVAILABLE:
    CATBOOST_PARAMS["task_type"] = "GPU"


def uses_gpu(model_name: str) -> bool:
    return {
        "lightgbm": LIGHTGBM_GPU_AVAILABLE,
        "xgboost": XGBOOST_GPU_AVAILABLE,
        "catboost": CATBOOST_GPU_AVAILABLE,
    }.get(model_name, False)


# ------------------------------------------------------------------
# Paralelisasi 28 horizon direct forecasting (dalam satu grup) via
# threading (satu proses, aman untuk manifest).
#
# PENTING (dipelajari dari crash run sebelumnya):
# - Model yang pakai GPU (CatBoost/XGBoost) WAJIB sequential (1 worker).
#   Konteks CUDA dalam satu proses tidak aman diminta beberapa thread
#   sekaligus (CatBoost: "device already requested 0" / crash kernel).
# - Model CPU (LightGBM, atau CatBoost/XGBoost saat GPU tidak aktif) memakai
#   jumlah worker DINAMIS berdasarkan ukuran grup, supaya total memory yang
#   dipakai salinan frame yang berjalan bersamaan tidak melebihi RAM.
# ------------------------------------------------------------------
PARALLEL_DIRECT_HORIZONS = True
N_HORIZON_WORKERS_CPU_MAX = min(8, os.cpu_count() or 4)  # batas ATAS untuk model CPU, bisa diturunkan otomatis per grup
N_HORIZON_WORKERS_GPU = 1                                 # WAJIB 1 - lihat catatan di atas, jangan dinaikkan tanpa memverifikasi versi library thread-safe

# Anggaran memory untuk salinan frame yang berjalan BERSAMAAN di direct
# forecasting (bukan total memory proses). Turunkan kalau masih OOM,
# naikkan kalau RAM masih banyak sisa dan ingin lebih banyak worker paralel.
MEMORY_BUDGET_BYTES_DIRECT = 2 * 1024**3  # 2 GB
DIRECT_FRAME_SAFETY_FACTOR = 2.5  # margin untuk train_fit/eval_hold copy + overhead pandas

if not PARALLEL_DIRECT_HORIZONS:
    N_HORIZON_WORKERS_CPU_MAX = 1
    N_HORIZON_WORKERS_GPU = 1


def compute_n_horizon_workers(model_name: str, train_group_df: pd.DataFrame) -> int:
    """Jumlah worker paralel untuk 28 horizon direct forecasting pada satu grup.
    GPU: selalu 1 (lihat catatan crash di atas). CPU: dihitung dari perkiraan
    ukuran memory satu salinan direct_frame, supaya beberapa worker paralel
    tidak pernah melebihi MEMORY_BUDGET_BYTES_DIRECT secara bersamaan."""
    if uses_gpu(model_name):
        return N_HORIZON_WORKERS_GPU

    approx_rows = len(train_group_df)
    approx_bytes_per_worker = approx_rows * len(FEATURES) * 4 * DIRECT_FRAME_SAFETY_FACTOR
    if approx_bytes_per_worker <= 0:
        return N_HORIZON_WORKERS_CPU_MAX

    budget_workers = max(1, int(MEMORY_BUDGET_BYTES_DIRECT // approx_bytes_per_worker))
    return max(1, min(N_HORIZON_WORKERS_CPU_MAX, budget_workers))


print("GENERAL_GPU_AVAILABLE:      ", GENERAL_GPU_AVAILABLE)
print("LIGHTGBM_GPU_AVAILABLE:     ", LIGHTGBM_GPU_AVAILABLE, "(False = wheel CPU-only, tetap jalan di CPU multi-thread)")
print("XGBOOST_GPU_AVAILABLE:      ", XGBOOST_GPU_AVAILABLE)
print("CATBOOST_GPU_AVAILABLE:     ", CATBOOST_GPU_AVAILABLE)
print("N_HORIZON_WORKERS_CPU_MAX:  ", N_HORIZON_WORKERS_CPU_MAX, "(atas; bisa turun otomatis per grup)")
print("N_HORIZON_WORKERS_GPU:      ", N_HORIZON_WORKERS_GPU, "(dipaksa sequential - lihat catatan crash)")
print("MEMORY_BUDGET_BYTES_DIRECT: ", f"{MEMORY_BUDGET_BYTES_DIRECT / 1024**3:.1f} GB")


GENERAL_GPU_AVAILABLE:       True
LIGHTGBM_GPU_AVAILABLE:      False (False = wheel CPU-only, tetap jalan di CPU multi-thread)
XGBOOST_GPU_AVAILABLE:       True
CATBOOST_GPU_AVAILABLE:      True
N_HORIZON_WORKERS_CPU_MAX:   8 (atas; bisa turun otomatis per grup)
N_HORIZON_WORKERS_GPU:       1 (dipaksa sequential - lihat catatan crash)
MEMORY_BUDGET_BYTES_DIRECT:  2.0 GB


## 3. Project Paths & Artefak

In [4]:
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)

DATASET_DIR = PROJECT_ROOT / "dataset"
FEATURE_DIR = DATASET_DIR / "features"

# Notebook 03 selalu menulis dengan MODE="full" (tidak ada varian smoke di
# level file), jadi sumber data notebook ini selalu direktori "_full_by_store".
# Subsampling untuk MODE="smoke" dilakukan di notebook ini sendiri (lihat sel
# pemuatan data), bukan dengan mengganti nama folder sumber.
FEATURE_SOURCE_MODE = "full"
TRAIN_PARQUET_DIR = FEATURE_DIR / f"train_features_{FEATURE_SOURCE_MODE}_by_store"
TEST_PARQUET_DIR = FEATURE_DIR / f"test_features_{FEATURE_SOURCE_MODE}_by_store"

assert TRAIN_PARQUET_DIR.exists(), f"Tidak ditemukan {TRAIN_PARQUET_DIR} - jalankan notebook 03 terlebih dahulu."
assert TEST_PARQUET_DIR.exists(), f"Tidak ditemukan {TEST_PARQUET_DIR} - jalankan notebook 03 terlebih dahulu."

MODEL_DIR = PROJECT_ROOT / "models" / MODE
RESULT_DIR = PROJECT_ROOT / "results" / MODE
PRED_DIR = RESULT_DIR / "predictions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = RESULT_DIR / "base_model_manifest.csv"
SUMMARY_PATH = RESULT_DIR / "model_training_summary.json"

print("PROJECT_ROOT:      ", PROJECT_ROOT)
print("TRAIN_PARQUET_DIR: ", TRAIN_PARQUET_DIR)
print("TEST_PARQUET_DIR:  ", TEST_PARQUET_DIR)
print("MODEL_DIR:         ", MODEL_DIR)
print("RESULT_DIR:        ", RESULT_DIR)
print("MANIFEST_PATH:     ", MANIFEST_PATH)

# ------------------------------------------------------------------
# CV Folds (Tahap 2 - notebook 01, expanding window CV0-CV3) - dipakai
# untuk melatih SEMUA base model dengan 4-fold time-series CV, alih-alih
# 1 main split tunggal. Validasi tiap fold diambil dari DALAM rentang
# training utama (d_1-d_1913), bukan menyentuh test_public (d_1914-d_1941)
# yang tetap disisihkan sebagai holdout akhir untuk Tahap 8.
# ------------------------------------------------------------------
SPLIT_DIR = DATASET_DIR / "splits"
CV_FOLDS_PATH = SPLIT_DIR / "cv_folds.csv"
assert CV_FOLDS_PATH.exists(), f"Tidak ditemukan {CV_FOLDS_PATH} - jalankan notebook 01 terlebih dahulu."

_cv_folds_df = pd.read_csv(
    CV_FOLDS_PATH,
    parse_dates=["training_start_date", "training_end_date", "validation_start_date", "validation_end_date"],
)

FOLD_SPECS = [
    {
        "fold": row["fold"],
        "train_end": row["training_end_date"],
        "valid_start": row["validation_start_date"],
        "valid_end": row["validation_end_date"],
    }
    for _, row in _cv_folds_df.iterrows()
]

print(f"CV_FOLDS_PATH: {CV_FOLDS_PATH}")
print(f"Jumlah fold CV: {len(FOLD_SPECS)}")
for fs in FOLD_SPECS:
    print(f"  {fs['fold']}: train <= {fs['train_end'].date()} | valid {fs['valid_start'].date()} .. {fs['valid_end'].date()}")


PROJECT_ROOT:       D:\speedrun-lulus
TRAIN_PARQUET_DIR:  D:\speedrun-lulus\dataset\features\train_features_full_by_store
TEST_PARQUET_DIR:   D:\speedrun-lulus\dataset\features\test_features_full_by_store
MODEL_DIR:          D:\speedrun-lulus\models\full
RESULT_DIR:         D:\speedrun-lulus\results\full
MANIFEST_PATH:      D:\speedrun-lulus\results\full\base_model_manifest.csv
CV_FOLDS_PATH: D:\speedrun-lulus\dataset\splits\cv_folds.csv
Jumlah fold CV: 4
  CV0: train <= 2016-01-03 | valid 2016-01-04 .. 2016-01-31
  CV1: train <= 2016-01-31 | valid 2016-02-01 .. 2016-02-28
  CV2: train <= 2016-02-28 | valid 2016-02-29 .. 2016-03-27
  CV3: train <= 2016-03-27 | valid 2016-03-28 .. 2016-04-24


## 4. Daftar Store & Kolom Fitur

Setiap file parquet sudah dipartisi per `store_id` (hasil notebook 03), sehingga strategi `byStore`,
`byStoreCategory`, dan `byStoreDepartment` semuanya adalah **subset di dalam satu file store** -
`byStoreCategory` = filter `cat_id` di dalam file store, `byStoreDepartment` = filter `dept_id` di dalam file
store. Karena itu setiap file store cukup dimuat **satu kali** dan dipakai ulang untuk ketiga strategi
grouping sekaligus, sebelum dibuang dari memory (`gc.collect()`) - konsisten dengan strategi hemat memory
pada notebook 03.

In [5]:
ALL_STORES = sorted(p.stem for p in TRAIN_PARQUET_DIR.glob("*.parquet"))
assert ALL_STORES, f"Tidak ada file parquet di {TRAIN_PARQUET_DIR}"

STORES = SMOKE_STORES if MODE == "smoke" else ALL_STORES
STORES = [s for s in STORES if s in ALL_STORES]

print("Store tersedia:", ALL_STORES)
print("Store dipakai pada MODE ini:", STORES)

# Ambil sampel kecil untuk mengetahui struktur kolom tanpa memuat seluruh file.
_sample = pd.read_parquet(TRAIN_PARQUET_DIR / f"{STORES[0]}.parquet").head(5)
print("\nJumlah kolom:", _sample.shape[1])
_sample.head()

Store tersedia: ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']
Store dipakai pada MODE ini: ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']

Jumlah kolom: 102


,id,item_id,dept_id,cat_id,store_id,state_id,store_cat,store_dept,date,d,wm_yr_wk,weekday,wday,dayofweek,month,year,is_weekend,event_name_1,event_name_2,event_type_1,...,rolling_mean_28,rolling_std_28,rolling_max_28,rolling_min_28,rolling_mean_60,rolling_std_60,rolling_max_60,rolling_min_60,rolling_mean_90,rolling_std_90,rolling_max_90,rolling_min_90,rolling_mean_180,rolling_std_180,rolling_max_180,rolling_min_180,rolling_mean_364,rolling_std_364,rolling_max_364,rolling_min_364
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,CA_1|FOODS,CA_1|FOODS_1,2011-01-29,d_1,11101,Saturday,1,5,1,2011,1,No_Event,No_Event,No_Type,...,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,CA_1|FOODS,CA_1|FOODS_1,2011-01-30,d_2,11101,Sunday,2,6,1,2011,1,No_Event,No_Event,No_Type,...,3.0,0.000000,3.0,3.0,3.0,0.000000,3.0,3.0,3.0,0.000000,3.0,3.0,3.0,0.000000,3.0,3.0,3.0,0.000000,3.0,3.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,CA_1|FOODS,CA_1|FOODS_1,2011-01-31,d_3,11101,Monday,3,0,1,2011,0,No_Event,No_Event,No_Type,...,1.5,2.121320,3.0,0.0,1.5,2.121320,3.0,0.0,1.5,2.121320,3.0,0.0,1.5,2.121320,3.0,0.0,1.5,2.121320,3.0,0.0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,CA_1|FOODS,CA_1|FOODS_1,2011-02-01,d_4,11101,Tuesday,4,1,2,2011,0,No_Event,No_Event,No_Type,...,1.0,1.732051,3.0,0.0,1.0,1.732051,3.0,0.0,1.0,1.732051,3.0,0.0,1.0,1.732051,3.0,0.0,1.0,1.732051,3.0,0.0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,CA_1|FOODS,CA_1|FOODS_1,2011-02-02,d_5,11101,Wednesday,5,2,2,2011,0,No_Event,No_Event,No_Type,...,1.0,1.414214,3.0,0.0,1.0,1.414214,3.0,0.0,1.0,1.414214,3.0,0.0,1.0,1.414214,3.0,0.0,1.0,1.414214,3.0,0.0


## 5. Target dan Feature Selection

In [6]:
TARGET = "sales"

# Kolom identitas/string mentah dan kolom administratif dikeluarkan dari FEATURES.
# Versi berlabel (_le) dan mean-encoded dari kolom-kolom ini sudah tersedia di
# dataset hasil preprocessing (notebook 02), jadi tidak ada informasi yang hilang.
RAW_ID_COLS = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "store_cat", "store_dept", "date", "d", "weekday",
    "event_name_1", "event_name_2", "event_type_1", "event_type_2",
]
EXCLUDE_COLS = RAW_ID_COLS + [TARGET, "split"]

FEATURES = [c for c in _sample.columns if c not in EXCLUDE_COLS]

print("Jumlah fitur:", len(FEATURES))
print(FEATURES)

Jumlah fitur: 85
['wm_yr_wk', 'wday', 'dayofweek', 'month', 'year', 'is_weekend', 'event_name_1_le', 'event_name_2_le', 'snap', 'sell_price', 'item_id_le', 'dept_id_le', 'cat_id_le', 'store_id_le', 'state_id_le', 'store_cat_le', 'store_dept_le', 'item_id_mean_enc', 'dept_id_mean_enc', 'cat_id_mean_enc', 'store_id_mean_enc', 'state_id_mean_enc', 'store_cat_mean_enc', 'store_dept_mean_enc', 'event_type_1__Cultural', 'event_type_1__National', 'event_type_1__No_Type', 'event_type_1__Religious', 'event_type_1__Sporting', 'event_type_2__Cultural', 'event_type_2__No_Type', 'event_type_2__Religious', 'week', 'day', 'days_to_event', 'price_mean', 'price_norm', 'price_cat_avg', 'price_cat_ratio', 'price_change', 'price_changed', 'is_discounted', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_35', 'lag_42', 'lag_49', 'lag_56', 'lag_63', 'lag_70', 'lag_77', 'lag_84', 'lag_90', 'lag_180', 'lag_364', 'rolling_mean_7', 'rolling_std_7', 'rolling_max_7', 'rolling_min_7', 'rolling_mean_14', 'rolling_std_14

## 6. Metrik Evaluasi

In [7]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    mask = y_true != 0
    if mask.sum() == 0:
        return float("nan")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def group_wrmsse_proxy(train_group_df: pd.DataFrame, eval_df: pd.DataFrame) -> float:
    '''Proxy WRMSSE pada level kelompok (bukan WRMSSE hierarkis resmi 12-level
    dari subbab 3.2.9.1). Bobot dinormalisasi supaya berjumlah 1 DI DALAM
    kelompok yang dievaluasi (dollar sales 28 hari terakhir training), dan scale
    dihitung dari naive one-step forecast pada training set tiap id. WRMSSE
    hierarkis penuh (mengagregasi ke 12 level dengan bobot dari SELURUH dataset)
    dihitung pada notebook evaluasi (Tahap 8), setelah semua kelompok selesai
    dilatih.

    PERBAIKAN (lihat catatan debugging setelah run LSTM): sebelumnya, saat total
    dollar_sales suatu kelompok <= 0 (mis. sell_price bernilai negatif/sentinel
    untuk sebagian SKU, atau kelompok sampel kecil tanpa penjualan berbayar di
    28 hari terakhir), kode lama jatuh ke bobot MENTAH (tidak dinormalisasi,
    bisa negatif) alih-alih proporsi 0-1 - itu yang menyebabkan WRMSSE_proxy
    LSTM keluar sebagai angka aneh seperti -714 atau -1004, padahal secara
    definisi RMSSE berbobot tidak pernah bisa negatif. Sekarang, setiap kali
    total bobot tidak positif, dipakai bobot SERAGAM (1/n) sebagai fallback
    yang aman, bukan nilai dollar_sales mentah.'''
    train_sorted = train_group_df.sort_values(["id", "date"])

    scales = (
        train_sorted.groupby("id")[TARGET]
        .apply(lambda s: np.mean(np.diff(s.values) ** 2) if len(s) > 1 else np.nan)
    )
    scales = scales.replace(0, np.nan)

    last28 = train_sorted.groupby("id").tail(28)
    dollar_sales = (last28[TARGET] * last28["sell_price"].fillna(0)).groupby(last28["id"]).sum()

    def _safe_normalize(s: pd.Series) -> pd.Series:
        '''Normalisasi ke proporsi 0-1. Kalau total <= 0 (termasuk NaN/negatif),
        kembalikan bobot SERAGAM alih-alih nilai mentah - mencegah bobot negatif
        atau tidak ternormalisasi ikut masuk ke perhitungan WRMSSE_proxy.'''
        total = s.sum()
        if pd.notna(total) and total > 0:
            return s / total
        n = len(s)
        return pd.Series(1.0 / n if n > 0 else np.nan, index=s.index)

    weights = _safe_normalize(dollar_sales)

    per_id_sq_err = (
        eval_df.assign(sq_err=(eval_df[TARGET] - eval_df["prediction"]) ** 2)
        .groupby("id")["sq_err"].mean()
    )

    common_ids = per_id_sq_err.index.intersection(scales.dropna().index).intersection(weights.index)
    if len(common_ids) == 0:
        return float("nan")

    rmsse_i = np.sqrt(per_id_sq_err.loc[common_ids] / scales.loc[common_ids])
    w = _safe_normalize(weights.loc[common_ids])
    return float((w * rmsse_i).sum())


def evaluate_group_predictions(train_group_df: pd.DataFrame, pred_df: pd.DataFrame) -> dict:
    return {
        "RMSE": rmse(pred_df[TARGET], pred_df["prediction"]),
        "MAE": mae(pred_df[TARGET], pred_df["prediction"]),
        "MAPE": mape(pred_df[TARGET], pred_df["prediction"]),
        "WRMSSE_proxy": group_wrmsse_proxy(train_group_df, pred_df),
        "n_series": pred_df["id"].nunique(),
        "n_rows": len(pred_df),
    }

print("Fungsi metrik siap: rmse, mae, mape, group_wrmsse_proxy, evaluate_group_predictions")

Fungsi metrik siap: rmse, mae, mape, group_wrmsse_proxy, evaluate_group_predictions


## 7. Checkpointing (Manifest)

Setiap kombinasi (`strategy`, `group_key`, `forecasting`, `model`) yang sudah selesai dilatih dicatat pada
`base_model_manifest.csv`. Menjalankan ulang notebook (mis. setelah crash atau restart) akan melewati baris
yang sudah tercatat, sehingga proses training bisa dilanjutkan tanpa mengulang dari awal.

In [8]:
MANIFEST_COLUMNS = [
    "fold", "strategy", "group_key", "model", "forecasting",
    "RMSE", "MAE", "MAPE", "WRMSSE_proxy", "n_series", "n_rows",
    "model_path", "prediction_path", "train_seconds",
]

def load_manifest() -> pd.DataFrame:
    if MANIFEST_PATH.exists():
        df = pd.read_csv(MANIFEST_PATH)
        if "fold" not in df.columns:
            # Manifest lama (sebelum CV diterapkan) hanya berisi hasil pada main
            # split (d_1-d_1913 vs d_1914-d_1941) - diberi label "MAIN" supaya
            # tetap tersimpan dan tidak tertukar dengan baris CV0-CV3 yang baru.
            df["fold"] = "MAIN"
        return df
    return pd.DataFrame(columns=MANIFEST_COLUMNS)


def is_done(manifest: pd.DataFrame, fold: str, strategy: str, group_key: str, model: str, forecasting: str) -> bool:
    if manifest.empty:
        return False
    return bool(
        (
            (manifest["fold"] == fold)
            & (manifest["strategy"] == strategy)
            & (manifest["group_key"] == group_key)
            & (manifest["model"] == model)
            & (manifest["forecasting"] == forecasting)
        ).any()
    )


def append_manifest_row(row: dict) -> None:
    """Ditulis atomik (temp file lalu os.replace) supaya kalau proses crash di
    tengah penulisan (seperti yang terjadi pada run sebelumnya karena error GPU),
    manifest.csv tidak pernah berakhir dalam keadaan setengah tertulis/corrupt -
    yang ada di disk selalu versi lengkap yang terakhir berhasil ditulis."""
    manifest = load_manifest()
    manifest = pd.concat([manifest, pd.DataFrame([row])], ignore_index=True)
    tmp_path = MANIFEST_PATH.with_suffix(".csv.tmp")
    manifest.to_csv(tmp_path, index=False)
    os.replace(tmp_path, MANIFEST_PATH)


manifest_preview = load_manifest()
print(f"Baris manifest yang sudah ada: {len(manifest_preview)}")
manifest_preview.tail()


Baris manifest yang sudah ada: 316


,fold,strategy,group_key,model,forecasting,RMSE,MAE,MAPE,WRMSSE_proxy,n_series,n_rows,model_path,prediction_path,train_seconds
311,CV0,byStoreDepartment,CA_2_HOBBIES_1,catboost,direct,2.232481,0.915434,58.083201,1.075547,416,11648,D:\speedrun-lulus\models\full\CV0__byStoreDepa...,D:\speedrun-lulus\results\full\predictions\CV0...,959.29
312,CV0,byStoreDepartment,CA_2_HOBBIES_2,lightgbm,recursive,0.785858,0.440502,69.216717,0.927699,149,4172,D:\speedrun-lulus\models\full\CV0__byStoreDepa...,D:\speedrun-lulus\results\full\predictions\CV0...,14.24
313,CV0,byStoreDepartment,CA_2_HOBBIES_2,xgboost,recursive,0.767437,0.413967,70.577266,0.941600,149,4172,D:\speedrun-lulus\models\full\CV0__byStoreDepa...,D:\speedrun-lulus\results\full\predictions\CV0...,5.69
314,CV0,byStoreDepartment,CA_2_HOBBIES_2,catboost,recursive,0.762715,0.429437,71.961266,0.948028,149,4172,D:\speedrun-lulus\models\full\CV0__byStoreDepa...,D:\speedrun-lulus\results\full\predictions\CV0...,10.42
315,CV0,byStoreDepartment,CA_2_HOBBIES_2,lightgbm,direct,0.749702,0.420547,68.290782,0.929855,149,4172,D:\speedrun-lulus\models\full\CV0__byStoreDepa...,D:\speedrun-lulus\results\full\predictions\CV0...,233.29


## 8. Direct Forecasting - Pembangunan Target per Horizon

In [9]:
def make_direct_training_frame(train_df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    '''Fitur pada hari t dipasangkan dengan target sales pada hari t+horizon,
    sesuai definisi direct forecasting (subbab 3.2.8.2): setiap model horizon-h
    memprediksi h hari ke depan langsung dari fitur historis aktual pada hari t,
    tanpa memakai hasil prediksi sebagai input.'''
    d = train_df.sort_values(["id", "date"]).copy()
    d["target"] = d.groupby("id")[TARGET].shift(-horizon)
    return d.dropna(subset=["target"])


def get_cutoff_features(train_df: pd.DataFrame) -> pd.DataFrame:
    '''Baris fitur pada hari terakhir training (cutoff, d_1913) per id -
    dipakai sebagai satu-satunya input untuk seluruh 28 model direct forecasting.'''
    last_date = train_df["date"].max()
    return train_df[train_df["date"] == last_date].drop_duplicates(subset=["id"]).copy()

print("Helper direct forecasting siap.")

Helper direct forecasting siap.


## 9. Recursive Forecasting - Rekomputasi Lag/Rolling Dinamis

Fitur `lag_7/14/21` dan `rolling_*_7/14/28` pada parquet hasil notebook 03 dihitung dari nilai `sales`
**aktual** (termasuk untuk hari-hari test, karena file `sales_train_evaluation.csv` M5 memang menyertakan
nilai aktual pada 28 hari tersebut). Memakai nilai itu apa adanya untuk recursive forecasting akan
membocorkan informasi masa depan. Sel ini merekomputasi fitur-fitur tersebut hari demi hari memakai buffer
riwayat (aktual sebelum cutoff, prediksi model sendiri setelah cutoff) - lag/rolling dengan window >= 28
(mis. lag_28, lag_364, rolling_mean_364) tidak pernah merujuk hari di dalam horizon 28 hari sehingga tetap
aman dipakai langsung dari parquet.

In [10]:
def recursive_forecast_group(model, train_group_df: pd.DataFrame, test_group_df: pd.DataFrame) -> pd.DataFrame:
    history: dict[str, list[float]] = {
        id_: list(g.sort_values("date")[TARGET].values[-400:])
        for id_, g in train_group_df.groupby("id")
    }

    predictions = []
    for date, day_rows in test_group_df.sort_values(["date", "id"]).groupby("date", sort=True):
        day_rows = day_rows.copy()

        for lag in SHORT_LAGS:
            day_rows[f"lag_{lag}"] = [
                history[i][-lag] if len(history[i]) >= lag else 0.0
                for i in day_rows["id"]
            ]

        for w in SHORT_ROLLING_WINDOWS:
            windows = [history[i][-w:] if history[i] else [0.0] for i in day_rows["id"]]
            day_rows[f"rolling_mean_{w}"] = [float(np.mean(x)) for x in windows]
            day_rows[f"rolling_std_{w}"] = [float(np.std(x)) if len(x) > 1 else 0.0 for x in windows]
            day_rows[f"rolling_max_{w}"] = [float(np.max(x)) for x in windows]
            day_rows[f"rolling_min_{w}"] = [float(np.min(x)) for x in windows]

        X_day = day_rows[FEATURES]
        y_hat = np.clip(model.predict(X_day), 0, None)
        day_rows["prediction"] = y_hat
        predictions.append(day_rows[["id", "date", TARGET, "prediction"]])

        for id_, pred in zip(day_rows["id"], y_hat):
            history[id_].append(float(pred))

    return pd.concat(predictions, ignore_index=True)

print("Helper recursive forecasting siap.")

Helper recursive forecasting siap.


## 10. Fungsi Generik Pelatihan Model Berbasis Pohon (per Kelompok)

In [11]:
def split_train_eval(df: pd.DataFrame, eval_days: int = EVAL_HOLD_DAYS):
    """eval_hold dipakai sebagai validation set untuk early stopping (subbab
    3.2.7, kolom early_stopping_rounds pada Tabel 3.4-3.6), diambil dari
    eval_days terakhir SEBELUM cutoff test set - tetap mengikuti prinsip
    time-based validation (tidak menyentuh data test)."""
    cutoff = df["date"].max() - pd.Timedelta(days=eval_days)
    train_fit = df[df["date"] <= cutoff]
    eval_hold = df[df["date"] > cutoff]
    if train_fit.empty or eval_hold.empty:
        return df, None
    return train_fit, eval_hold


def instantiate_model(model_name: str, n_jobs_override: int | None = None):
    """n_jobs_override dipakai saat beberapa horizon direct forecasting
    dilatih paralel dalam thread yang berbeda (lihat train_direct_tree) - supaya
    total thread yang dipakai tidak melebihi jumlah core (oversubscription)."""
    if model_name == "lightgbm":
        params = dict(LIGHTGBM_PARAMS)
        if n_jobs_override is not None:
            params["n_jobs"] = n_jobs_override
        return LGBMRegressor(**params)
    if model_name == "xgboost":
        params = dict(XGBOOST_PARAMS)
        if n_jobs_override is not None:
            params["n_jobs"] = n_jobs_override
        return XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    if model_name == "catboost":
        params = dict(CATBOOST_PARAMS)
        if n_jobs_override is not None:
            params["thread_count"] = n_jobs_override
        return CatBoostRegressor(**params)
    raise ValueError(f"Model tree tidak dikenal: {model_name}")


def fit_model(model_name: str, model, X_train, y_train, X_eval=None, y_eval=None):
    if X_eval is None:
        model.fit(X_train, y_train)
        return model

    if model_name == "lightgbm":
        from lightgbm import early_stopping, log_evaluation
        model.fit(
            X_train, y_train,
            eval_set=[(X_eval, y_eval)],
            callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)],
        )
    elif model_name == "xgboost":
        model.fit(X_train, y_train, eval_set=[(X_eval, y_eval)], verbose=False)
    elif model_name == "catboost":
        model.fit(
            X_train, y_train,
            eval_set=(X_eval, y_eval),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )
    return model


def save_tree_model(model, model_name: str, path: Path) -> None:
    joblib.dump(model, path)


def train_recursive_tree(fold: str, model_name: str, strategy: str, group_key: str,
                          train_group_df: pd.DataFrame, test_group_df: pd.DataFrame) -> dict:
    train_fit, eval_hold = split_train_eval(train_group_df)
    model = instantiate_model(model_name)
    if eval_hold is not None:
        model = fit_model(model_name, model, train_fit[FEATURES], train_fit[TARGET],
                           eval_hold[FEATURES], eval_hold[TARGET])
    else:
        model = fit_model(model_name, model, train_group_df[FEATURES], train_group_df[TARGET])

    pred_df = recursive_forecast_group(model, train_group_df, test_group_df)

    tag = f"{fold}__{strategy}__{group_key}__{model_name}__recursive"
    model_path = MODEL_DIR / f"{tag}.pkl"
    pred_path = PRED_DIR / f"{tag}.csv"
    save_tree_model(model, model_name, model_path)
    pred_df.to_csv(pred_path, index=False)

    metrics = evaluate_group_predictions(train_group_df, pred_df)
    metrics.update({"model_path": str(model_path), "prediction_path": str(pred_path)})
    return metrics


def _fit_direct_horizon(fold: str, model_name: str, strategy: str, group_key: str,
                         train_group_df: pd.DataFrame, cutoff_features: pd.DataFrame,
                         h_idx: int, horizon_date, n_jobs_override: int | None):
    """Satu horizon direct forecasting - fungsi ini dipanggil paralel (lihat
    train_direct_tree) karena setiap horizon sepenuhnya independen dari horizon
    lainnya (subbab 3.2.8.2: setiap model horizon-h dilatih dan diterapkan
    sendiri-sendiri)."""
    direct_frame = make_direct_training_frame(train_group_df, h_idx)
    if direct_frame.empty:
        return None

    train_fit, eval_hold = split_train_eval(direct_frame)
    model = instantiate_model(model_name, n_jobs_override=n_jobs_override)
    if eval_hold is not None and len(eval_hold) > 0:
        model = fit_model(model_name, model, train_fit[FEATURES], train_fit["target"],
                           eval_hold[FEATURES], eval_hold["target"])
    else:
        model = fit_model(model_name, model, direct_frame[FEATURES], direct_frame["target"])

    X_pred = cutoff_features[FEATURES]
    y_hat = np.clip(model.predict(X_pred), 0, None)

    horizon_pred = pd.DataFrame({
        "id": cutoff_features["id"].values,
        "date": horizon_date,
        "prediction": y_hat,
    })

    tag = f"{fold}__{strategy}__{group_key}__{model_name}__direct__h{h_idx:02d}"
    model_path = MODEL_DIR / f"{tag}.pkl"
    save_tree_model(model, model_name, model_path)

    del model, direct_frame, train_fit, eval_hold
    return horizon_pred, str(model_path)


def train_direct_tree(fold: str, model_name: str, strategy: str, group_key: str,
                       train_group_df: pd.DataFrame, test_group_df: pd.DataFrame) -> dict:
    cutoff_features = get_cutoff_features(train_group_df)
    test_sorted = test_group_df.sort_values(["id", "date"])
    test_dates = sorted(test_sorted["date"].unique())[:FORECAST_HORIZON]

    n_workers = compute_n_horizon_workers(model_name, train_group_df)
    # Bagi rata core CPU antar worker paralel supaya tiap fit() individual tidak
    # ikut memakai semua core (oversubscription) saat beberapa horizon jalan bersamaan.
    n_jobs_override = max(1, (os.cpu_count() or 4) // n_workers)

    results = Parallel(n_jobs=n_workers, backend="threading")(
        delayed(_fit_direct_horizon)(
            fold, model_name, strategy, group_key, train_group_df, cutoff_features,
            h_idx, horizon_date, n_jobs_override,
        )
        for h_idx, horizon_date in enumerate(test_dates, start=1)
    )

    all_preds, horizon_model_paths = [], []
    for r in results:
        if r is None:
            continue
        horizon_pred, model_path = r
        all_preds.append(horizon_pred)
        horizon_model_paths.append(model_path)
    gc.collect()

    pred_df = pd.concat(all_preds, ignore_index=True)
    pred_df = pred_df.merge(test_sorted[["id", "date", TARGET]], on=["id", "date"], how="left")

    tag = f"{fold}__{strategy}__{group_key}__{model_name}__direct"
    pred_path = PRED_DIR / f"{tag}.csv"
    pred_df.to_csv(pred_path, index=False)

    metrics = evaluate_group_predictions(train_group_df, pred_df)
    metrics.update({
        "model_path": ";".join(horizon_model_paths),
        "prediction_path": str(pred_path),
    })
    return metrics

print("Fungsi pelatihan tree model (recursive & direct, direct dijalankan paralel per horizon) siap.")


Fungsi pelatihan tree model (recursive & direct, direct dijalankan paralel per horizon) siap.


## 11. Loop Utama - Model Berbasis Pohon (LightGBM, XGBoost, CatBoost)

Loop berjalan **per store** (memuat file parquet store sekali), lalu untuk setiap strategi grouping
menurunkan sub-kelompok di dalam store tersebut (`byStore` = seluruh store, `byStoreCategory` = per `cat_id`,
`byStoreDepartment` = per `dept_id`). Kombinasi yang sudah ada di manifest dilewati (checkpoint/resume).

In [12]:
def iter_groups_in_store(store_train: pd.DataFrame, store_test: pd.DataFrame, strategy: str):
    key_cols = GROUPING_STRATEGIES[strategy]
    if strategy == "byStore":
        yield store_train["store_id"].iloc[0], store_train, store_test
        return
    combos = store_train[key_cols].drop_duplicates().itertuples(index=False, name=None)
    for combo in combos:
        mask_train = np.logical_and.reduce([store_train[c] == v for c, v in zip(key_cols, combo)])
        mask_test = np.logical_and.reduce([store_test[c] == v for c, v in zip(key_cols, combo)])
        group_key = "_".join(str(v) for v in combo)
        yield group_key, store_train[mask_train], store_test[mask_test]


run_start = time.time()
n_trained_this_run = 0

# CATATAN CV: setiap store dibaca SEKALI dari parquet (hemat I/O), lalu untuk
# tiap fold CV (CV0-CV3) di-slice berdasarkan tanggal SAJA dari frame yang
# sama - validasi tiap fold datang dari dalam rentang training utama
# (d_1-d_1913), test_public (d_1914-d_1941) tidak disentuh di loop ini
# (baris manifest lama berlabel fold="MAIN" tetap tersimpan terpisah).
for store in tqdm(STORES, desc="Store"):
    store_train_full = pd.read_parquet(TRAIN_PARQUET_DIR / f"{store}.parquet")
    store_train_full["date"] = pd.to_datetime(store_train_full["date"])

    if MODE == "smoke" and SMOKE_MAX_ITEMS_PER_STORE:
        keep_ids = store_train_full["id"].drop_duplicates().head(SMOKE_MAX_ITEMS_PER_STORE)
        store_train_full = store_train_full[store_train_full["id"].isin(keep_ids)]

    for fold_spec in tqdm(FOLD_SPECS, desc=f"Fold ({store})", leave=False):
        fold = fold_spec["fold"]
        fold_train = store_train_full[store_train_full["date"] <= fold_spec["train_end"]]
        fold_valid = store_train_full[
            (store_train_full["date"] >= fold_spec["valid_start"])
            & (store_train_full["date"] <= fold_spec["valid_end"])
        ]
        if fold_train.empty or fold_valid.empty:
            continue

        manifest = load_manifest()

        for strategy in GROUPING_STRATEGIES:
            for group_key, group_train, group_test in iter_groups_in_store(fold_train, fold_valid, strategy):
                if group_train.empty or group_test.empty:
                    continue
                for forecasting in FORECASTING_STRATEGIES:
                    for model_name in TREE_BASE_MODELS:
                        if is_done(manifest, fold, strategy, group_key, model_name, forecasting):
                            continue

                        t0 = time.time()
                        try:
                            if forecasting == "recursive":
                                metrics = train_recursive_tree(fold, model_name, strategy, group_key, group_train, group_test)
                            else:
                                metrics = train_direct_tree(fold, model_name, strategy, group_key, group_train, group_test)
                        except Exception as exc:  # pragma: no cover - defensive logging
                            print(f"[GAGAL] {fold}/{strategy}/{group_key}/{model_name}/{forecasting}: {exc}")
                            continue

                        metrics.update({
                            "fold": fold,
                            "strategy": strategy,
                            "group_key": group_key,
                            "model": model_name,
                            "forecasting": forecasting,
                            "train_seconds": round(time.time() - t0, 2),
                        })
                        append_manifest_row(metrics)
                        manifest = load_manifest()
                        n_trained_this_run += 1

        del fold_train, fold_valid
        gc.collect()

    del store_train_full
    gc.collect()

print(f"\nSelesai. {n_trained_this_run} kombinasi baru dilatih pada run ini "
      f"({round(time.time() - run_start, 1)} detik).")
print(f"Total baris manifest sekarang: {len(load_manifest())}")


Store:   0%|          | 0/10 [00:00<?, ?it/s]

Fold (CA_1):   0%|          | 0/4 [00:00<?, ?it/s]

Fold (CA_2):   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 12. LSTM Global - Konstruksi Sekuens

Sesuai subbab 3.2.7.3, LSTM dilatih secara global (bukan per kelompok) dengan window sekuens 28 hari.
Fitur temporal (riwayat `sales`) dibentuk lewat sliding window; fitur lain (kalender, harga, lag/rolling
jangka panjang, dan identitas hierarkis) diperlakukan sebagai *static contextual features* dan di-tile ke
setiap timestep, sesuai subbab 3.2.7.3d.

In [ ]:
LSTM_TEMPORAL_COL = TARGET
LSTM_STATIC_FEATURES = [c for c in FEATURES if c != LSTM_TEMPORAL_COL]


def load_full_train_test_for_lstm(stores: list[str], max_series: int | None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Dimuat & disubsample PER STORE (bukan concat semua store dulu baru
    disubsample). Sebelumnya seluruh parquet train+test untuk SEMUA store
    dibaca penuh ke memory sebelum difilter ke max_series - untuk laptop
    dengan RAM terbatas, itu sendiri sudah bisa OOM sebelum sampai ke
    tahap pembangunan sequence. Dengan memfilter per-store, hanya satu
    frame store penuh yang pernah ada di memory di satu waktu."""
    rng = np.random.default_rng(SEED)
    n_stores = len(stores)
    quota_per_store = None if max_series is None else max(1, max_series // n_stores)

    train_parts, test_parts = [], []
    for store in stores:
        store_train = pd.read_parquet(TRAIN_PARQUET_DIR / f"{store}.parquet")
        store_test = pd.read_parquet(TEST_PARQUET_DIR / f"{store}.parquet")

        if quota_per_store is not None:
            ids = store_train["id"].unique()
            if len(ids) > quota_per_store:
                keep_ids = rng.choice(ids, size=quota_per_store, replace=False)
                store_train = store_train[store_train["id"].isin(keep_ids)]
                store_test = store_test[store_test["id"].isin(keep_ids)]

        store_train["date"] = pd.to_datetime(store_train["date"])
        store_test["date"] = pd.to_datetime(store_test["date"])
        train_parts.append(store_train)
        test_parts.append(store_test)
        del store_train, store_test
        gc.collect()

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    del train_parts, test_parts
    gc.collect()
    return train_df, test_df


def build_lstm_sequences(df: pd.DataFrame, window: int, target_offset: int = 1, stride: int = 1,
                          max_samples: int | None = None, seed: int = SEED):
    """Bangun sequence LSTM dengan DUA input terpisah alih-alih satu tensor
    gabungan:
      - X_seq   : (n, window, 1)      -> riwayat sales (temporal, berubah tiap hari)
      - X_static: (n, n_static_feat)  -> fitur kontekstual PADA HARI CUTOFF SAJA

    Perbaikan RAM utama dibanding versi sebelumnya: versi sebelumnya
    men-tile (mengulang) seluruh fitur statis ke SETIAP timestep window
    (window x n_features per sample), padahal fitur statis tidak berubah
    dalam satu window - itu membuang memory ~window x lipat (28x) tanpa
    guna, dan menjadi penyebab utama OOM. Di sini fitur statis disimpan
    HANYA SEKALI per sample dan digabung ke output LSTM di dalam model
    (lihat build_lstm_model), bukan di data.

    `stride` melompati hari antar training sample (mengurangi jumlah
    sample ~stride x, lihat LSTM_SEQ_STRIDE). `max_samples` melakukan
    subsampling acak tambahan kalau total sample (setelah stride) masih
    melebihi batas RAM yang aman."""
    X_seq_list, X_static_list, y_vals, meta_ids, meta_dates = [], [], [], [], []
    for id_, g in df.sort_values(["id", "date"]).groupby("id"):
        sales = g[LSTM_TEMPORAL_COL].values.astype("float32")
        static = g[LSTM_STATIC_FEATURES].values.astype("float32")
        dates = g["date"].values
        n = len(g)
        for t in range(window, n - target_offset + 1, stride):
            X_seq_list.append(sales[t - window:t].reshape(window, 1))
            X_static_list.append(static[t - 1])  # snapshot statis di hari cutoff, TIDAK di-tile
            y_vals.append(sales[t + target_offset - 1])
            meta_ids.append(id_)
            meta_dates.append(dates[t + target_offset - 1])

    if not X_seq_list:
        return (np.empty((0, window, 1), dtype="float32"),
                np.empty((0, len(LSTM_STATIC_FEATURES)), dtype="float32"),
                np.empty((0,), dtype="float32"), [], [])

    X_seq = np.stack(X_seq_list).astype("float32")
    X_static = np.stack(X_static_list).astype("float32")
    y = np.array(y_vals, dtype="float32")
    del X_seq_list, X_static_list

    if max_samples is not None and len(X_seq) > max_samples:
        rng = np.random.default_rng(seed)
        keep = rng.choice(len(X_seq), size=max_samples, replace=False)
        keep.sort()
        X_seq = X_seq[keep]
        X_static = X_static[keep]
        y = y[keep]
        meta_ids = [meta_ids[i] for i in keep]
        meta_dates = [meta_dates[i] for i in keep]

    gc.collect()
    return X_seq, X_static, y, meta_ids, meta_dates


print("LSTM_STATIC_FEATURES:", len(LSTM_STATIC_FEATURES), "fitur statis + 1 fitur temporal (sales).")
print(f"LSTM_MAX_SERIES pada MODE={MODE!r}:", LSTM_MAX_SERIES)
print(f"LSTM_SEQ_STRIDE: {LSTM_SEQ_STRIDE} | LSTM_MAX_TRAIN_SAMPLES: {LSTM_MAX_TRAIN_SAMPLES}")


LSTM_STATIC_FEATURES: 85 fitur statis + 1 fitur temporal (sales).
LSTM_MAX_SERIES pada MODE='full': 1500
LSTM_SEQ_STRIDE: 7 | LSTM_MAX_TRAIN_SAMPLES: 250000


## 13. LSTM - Pelatihan Recursive & Direct

In [ ]:
def build_lstm_model(n_static_features: int) -> Model:
    """Arsitektur dua-input: cabang LSTM murni untuk riwayat sales (temporal),
    cabang Dense untuk fitur statis, digabung (concat) setelah LSTM - BUKAN
    di-tile ke tiap timestep seperti versi sebelumnya. Ini menghemat memory
    baik saat membangun data (lihat build_lstm_sequences) maupun saat model
    memproses batch (tensor input LSTM jadi (batch, window, 1) alih-alih
    (batch, window, 1+n_static), jauh lebih kecil untuk VRAM 4 GB)."""
    seq_input = Input(shape=(LSTM_PARAMS["window_size"], 1), name="seq_input")
    static_input = Input(shape=(n_static_features,), name="static_input")

    x = LSTM(LSTM_PARAMS["lstm_units_1"], return_sequences=True)(seq_input)
    x = Dropout(LSTM_PARAMS["dropout_rate"])(x)
    x = LSTM(LSTM_PARAMS["lstm_units_2"], return_sequences=False)(x)
    x = Dropout(LSTM_PARAMS["dropout_rate"])(x)

    x = Concatenate()([x, static_input])
    x = Dense(LSTM_PARAMS["dense_units_1"], activation="relu")(x)
    x = Dense(LSTM_PARAMS["dense_units_2"], activation="relu")(x)
    out = Dense(1, activation="linear")(x)

    model = Model(inputs=[seq_input, static_input], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LSTM_PARAMS["learning_rate"]),
        loss="mse",
    )
    return model


def fit_lstm(X_seq, X_static, y, X_seq_val=None, X_static_val=None, y_val=None):
    model = build_lstm_model(X_static.shape[-1])
    has_val = X_seq_val is not None
    callbacks = [
        EarlyStopping(monitor="val_loss" if has_val else "loss",
                       patience=LSTM_PARAMS["early_stopping_patience"], restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss" if has_val else "loss", patience=3, factor=0.5),
    ]
    model.fit(
        [X_seq, X_static], y,
        validation_data=([X_seq_val, X_static_val], y_val) if has_val else None,
        epochs=LSTM_PARAMS["epochs"],
        batch_size=LSTM_PARAMS["batch_size"],
        callbacks=callbacks,
        verbose=0,
    )
    return model


manifest = load_manifest()
# Dimuat SEKALI untuk seluruh d_1-d_1913 (lstm_train_df, disubsample ke
# LSTM_MAX_SERIES id) - tiap fold CV cukup slice tanggal dari frame ini, tidak
# perlu load ulang parquet. lstm_test_df (d_1914-d_1941, holdout asli) tetap
# dimuat untuk kompatibilitas manifest lama (fold="MAIN") tapi TIDAK dipakai
# di loop CV di bawah.
lstm_train_df, lstm_test_df = load_full_train_test_for_lstm(STORES, LSTM_MAX_SERIES)

window = LSTM_PARAMS["window_size"]

print("Data LSTM disiapkan:", lstm_train_df["id"].nunique(), "deret waktu.")
print("NOTE: scaler fitur statis di-fit PER FOLD (lihat loop CV berikutnya), bukan sekali global,")
print("supaya tidak ada informasi dari validation fold yang bocor ke scaler training fold tersebut.")


Data LSTM disiapkan: 1500 deret waktu.


In [ ]:
# ------------------------- LSTM Recursive (per fold CV) -------------------------
for fold_spec in FOLD_SPECS:
    fold = fold_spec["fold"]
    if is_done(manifest, fold, "global", "all", "lstm", "recursive"):
        print(f"[{fold}] LSTM recursive sudah ada di manifest, dilewati.")
        continue

    t0 = time.time()
    fold_train_raw = lstm_train_df[lstm_train_df["date"] <= fold_spec["train_end"]].copy()
    fold_valid_raw = lstm_train_df[
        (lstm_train_df["date"] >= fold_spec["valid_start"])
        & (lstm_train_df["date"] <= fold_spec["valid_end"])
    ].copy()
    if fold_train_raw.empty or fold_valid_raw.empty:
        print(f"[{fold}] dilewati - data train/valid kosong untuk fold ini.")
        continue

    # scaler di-fit HANYA dari training fold ini (mencegah leakage antar fold).
    fold_scaler = StandardScaler()
    fold_train_raw[LSTM_STATIC_FEATURES] = fold_scaler.fit_transform(fold_train_raw[LSTM_STATIC_FEATURES])
    fold_valid_raw[LSTM_STATIC_FEATURES] = fold_scaler.transform(fold_valid_raw[LSTM_STATIC_FEATURES])
    joblib.dump(fold_scaler, MODEL_DIR / f"{fold}__lstm_static_scaler_recursive.pkl")

    X_seq_all, X_static_all, y_all, ids_all, dates_all = build_lstm_sequences(
        fold_train_raw, window, target_offset=1,
        stride=LSTM_SEQ_STRIDE, max_samples=LSTM_MAX_TRAIN_SAMPLES,
    )
    n_val = max(1, int(0.1 * len(X_seq_all)))
    X_seq_fit, X_static_fit, y_fit = X_seq_all[:-n_val], X_static_all[:-n_val], y_all[:-n_val]
    X_seq_val, X_static_val, y_val = X_seq_all[-n_val:], X_static_all[-n_val:], y_all[-n_val:]
    del X_seq_all, X_static_all, y_all
    gc.collect()

    lstm_recursive_model = fit_lstm(X_seq_fit, X_static_fit, y_fit, X_seq_val, X_static_val, y_val)
    model_path = MODEL_DIR / f"{fold}__global__lstm__recursive.keras"
    lstm_recursive_model.save(model_path)
    del X_seq_fit, X_static_fit, y_fit, X_seq_val, X_static_val, y_val
    gc.collect()

    # --- recursive forecast pada validation window fold ini ---
    history = {
        id_: list(g.sort_values("date")[LSTM_TEMPORAL_COL].values[-400:])
        for id_, g in fold_train_raw.groupby("id")
    }

    preds_rows = []
    test_sorted = fold_valid_raw.sort_values(["date", "id"])
    for step, (date, day_rows) in enumerate(test_sorted.groupby("date", sort=True)):
        seq_batch, static_batch, ids_batch = [], [], []
        for _, row in day_rows.iterrows():
            id_ = row["id"]
            seq_sales = np.array(history[id_][-window:], dtype="float32").reshape(window, 1)
            static_vec = row[LSTM_STATIC_FEATURES].values.astype("float32")
            seq_batch.append(seq_sales)
            static_batch.append(static_vec)
            ids_batch.append(id_)
        seq_batch = np.stack(seq_batch)
        static_batch = np.stack(static_batch)
        y_hat = np.clip(lstm_recursive_model.predict([seq_batch, static_batch], verbose=0).flatten(), 0, None)
        preds_rows.append(pd.DataFrame({"id": ids_batch, "date": date, "prediction": y_hat}))
        for id_, pred in zip(ids_batch, y_hat):
            history[id_].append(float(pred))

    pred_df = pd.concat(preds_rows, ignore_index=True)
    pred_df = pred_df.merge(fold_valid_raw[["id", "date", TARGET]], on=["id", "date"], how="left")
    pred_path = PRED_DIR / f"{fold}__global__lstm__recursive.csv"
    pred_df.to_csv(pred_path, index=False)

    metrics = evaluate_group_predictions(fold_train_raw, pred_df)
    metrics.update({
        "fold": fold, "strategy": "global", "group_key": "all", "model": "lstm", "forecasting": "recursive",
        "model_path": str(model_path),
        "prediction_path": str(pred_path),
        "train_seconds": round(time.time() - t0, 2),
    })
    append_manifest_row(metrics)
    manifest = load_manifest()
    print(f"[{fold}] LSTM recursive selesai:", metrics)

    del lstm_recursive_model, history, preds_rows, pred_df, fold_train_raw, fold_valid_raw
    gc.collect()


LSTM recursive sudah ada di manifest, dilewati.


In [ ]:
# ------------------------- LSTM Direct (per fold CV) -------------------------
for fold_spec in FOLD_SPECS:
    fold = fold_spec["fold"]
    if is_done(manifest, fold, "global", "all", "lstm", "direct"):
        print(f"[{fold}] LSTM direct sudah ada di manifest, dilewati.")
        continue

    t0 = time.time()
    fold_train_raw = lstm_train_df[lstm_train_df["date"] <= fold_spec["train_end"]].copy()
    fold_valid_raw = lstm_train_df[
        (lstm_train_df["date"] >= fold_spec["valid_start"])
        & (lstm_train_df["date"] <= fold_spec["valid_end"])
    ].copy()
    if fold_train_raw.empty or fold_valid_raw.empty:
        print(f"[{fold}] dilewati - data train/valid kosong untuk fold ini.")
        continue

    fold_scaler = StandardScaler()
    fold_train_raw[LSTM_STATIC_FEATURES] = fold_scaler.fit_transform(fold_train_raw[LSTM_STATIC_FEATURES])
    fold_valid_raw[LSTM_STATIC_FEATURES] = fold_scaler.transform(fold_valid_raw[LSTM_STATIC_FEATURES])
    joblib.dump(fold_scaler, MODEL_DIR / f"{fold}__lstm_static_scaler_direct.pkl")

    test_dates_sorted = sorted(fold_valid_raw["date"].unique())[:FORECAST_HORIZON]
    all_preds = []
    horizon_paths = []

    for h_idx, horizon_date in enumerate(test_dates_sorted, start=1):
        X_seq_h, X_static_h, y_h, ids_h, dates_h = build_lstm_sequences(
            fold_train_raw, window, target_offset=h_idx,
            stride=LSTM_SEQ_STRIDE, max_samples=LSTM_MAX_TRAIN_SAMPLES,
        )
        if len(X_seq_h) == 0:
            continue
        n_val = max(1, int(0.1 * len(X_seq_h)))
        X_seq_fit, X_static_fit, y_fit = X_seq_h[:-n_val], X_static_h[:-n_val], y_h[:-n_val]
        X_seq_val, X_static_val, y_val = X_seq_h[-n_val:], X_static_h[-n_val:], y_h[-n_val:]
        del X_seq_h, X_static_h, y_h

        model_h = fit_lstm(X_seq_fit, X_static_fit, y_fit, X_seq_val, X_static_val, y_val)
        model_path = MODEL_DIR / f"{fold}__global__lstm__direct__h{h_idx:02d}.keras"
        model_h.save(model_path)
        horizon_paths.append(str(model_path))

        cutoff_rows = fold_train_raw.sort_values("date").groupby("id").tail(window)
        seq_cutoff, static_cutoff, ids_cutoff = [], [], []
        for id_, g in cutoff_rows.groupby("id"):
            if len(g) < window:
                continue
            seq_sales = g[LSTM_TEMPORAL_COL].values.astype("float32").reshape(window, 1)
            static_vec = g[LSTM_STATIC_FEATURES].values[-1].astype("float32")
            seq_cutoff.append(seq_sales)
            static_cutoff.append(static_vec)
            ids_cutoff.append(id_)
        seq_cutoff = np.stack(seq_cutoff)
        static_cutoff = np.stack(static_cutoff)
        y_hat = np.clip(model_h.predict([seq_cutoff, static_cutoff], verbose=0).flatten(), 0, None)

        all_preds.append(pd.DataFrame({"id": ids_cutoff, "date": horizon_date, "prediction": y_hat}))

        del model_h, X_seq_fit, X_static_fit, y_fit, X_seq_val, X_static_val, y_val, seq_cutoff, static_cutoff
        gc.collect()

    pred_df = pd.concat(all_preds, ignore_index=True)
    pred_df = pred_df.merge(fold_valid_raw[["id", "date", TARGET]], on=["id", "date"], how="left")
    pred_path = PRED_DIR / f"{fold}__global__lstm__direct.csv"
    pred_df.to_csv(pred_path, index=False)

    metrics = evaluate_group_predictions(fold_train_raw, pred_df)
    metrics.update({
        "fold": fold, "strategy": "global", "group_key": "all", "model": "lstm", "forecasting": "direct",
        "model_path": ";".join(horizon_paths),
        "prediction_path": str(pred_path),
        "train_seconds": round(time.time() - t0, 2),
    })
    append_manifest_row(metrics)
    manifest = load_manifest()
    print(f"[{fold}] LSTM direct selesai:", metrics)

    del fold_train_raw, fold_valid_raw, pred_df
    gc.collect()

del lstm_train_df, lstm_test_df
gc.collect()


LSTM direct sudah ada di manifest, dilewati.


0

## 14. Konsolidasi Hasil Seluruh Konfigurasi

In [ ]:
results_df = load_manifest()
print("Total baris manifest (fold x strategi x kelompok x model x forecasting):", len(results_df))
print("\nJumlah baris per fold:")
display(results_df["fold"].value_counts())

summary_by_config = (
    results_df
    .groupby(["fold", "strategy", "model", "forecasting"])
    .agg(
        n_groups=("group_key", "nunique"),
        mean_RMSE=("RMSE", "mean"),
        mean_MAE=("MAE", "mean"),
        mean_MAPE=("MAPE", "mean"),
        mean_WRMSSE_proxy=("WRMSSE_proxy", "mean"),
        total_train_seconds=("train_seconds", "sum"),
    )
    .reset_index()
    .sort_values(["fold", "mean_WRMSSE_proxy"])
)

summary_by_config


Total kombinasi (strategi x kelompok x model x forecasting) tercatat: 662


,strategy,model,forecasting,n_groups,mean_RMSE,mean_MAE,mean_MAPE,mean_WRMSSE_proxy,total_train_seconds
19,global,lstm,recursive,1,2.299767,1.179915,60.741787,-18399.409256,244.67
18,global,lstm,direct,1,2.244975,1.065871,66.204522,-16885.698278,6681.60
9,byStoreCategory,lightgbm,recursive,30,1.833065,0.964347,58.309119,0.966504,2637.75
15,byStoreDepartment,lightgbm,recursive,70,1.755829,0.966779,59.757326,0.972676,2272.35
8,byStoreCategory,lightgbm,direct,30,1.844594,0.949439,58.010617,0.973293,55801.27
3,byStore,lightgbm,recursive,10,2.056645,1.051301,56.665911,0.973763,3273.20
11,byStoreCategory,xgboost,recursive,30,1.861638,0.945222,62.282050,0.976424,1023.82
10,byStoreCategory,xgboost,direct,30,1.859419,0.948231,57.714550,0.976632,23520.83
2,byStore,lightgbm,direct,10,2.062733,1.033109,56.062080,0.977543,74871.16
14,byStoreDepartment,lightgbm,direct,70,1.763554,0.953314,59.613966,0.978517,40368.53


## 15. Simpan Ringkasan Akhir

In [ ]:
n_cv_folds = len(FOLD_SPECS)
expected_tree_configs_per_fold = len(GROUPING_STRATEGIES) * len(TREE_BASE_MODELS) * len(FORECASTING_STRATEGIES)
expected_lstm_configs_per_fold = 1 * 1 * len(FORECASTING_STRATEGIES)  # global, tidak per kelompok
expected_tree_configs = expected_tree_configs_per_fold * n_cv_folds
expected_lstm_configs = expected_lstm_configs_per_fold * n_cv_folds

summary = {
    "source": "Outline Skripsi Bab 3 - Tahap 6/7: Strategi Grouped Modeling & Model Training (Base Models)",
    "mode": MODE,
    "cv_folds": [fs["fold"] for fs in FOLD_SPECS],
    "grouping_strategies": {k: v for k, v in GROUPING_STRATEGIES.items()},
    "forecasting_strategies": FORECASTING_STRATEGIES,
    "base_models": BASE_MODELS,
    "stores_used": STORES,
    "expected_tree_config_rows": expected_tree_configs,
    "expected_lstm_config_rows": expected_lstm_configs,
    "expected_tree_config_rows_per_fold": expected_tree_configs_per_fold,
    "expected_lstm_config_rows_per_fold": expected_lstm_configs_per_fold,
    "n_manifest_rows": int(len(results_df)),
    "hyperparameters": {
        "lightgbm": LIGHTGBM_PARAMS,
        "xgboost": XGBOOST_PARAMS,
        "catboost": CATBOOST_PARAMS,
        "lstm": LSTM_PARAMS,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    },
    "notes": [
        "Tree-based models (LightGBM/XGBoost/CatBoost) dilatih terpisah per kelompok pada tiga strategi "
        "grouped modeling (subbab 3.2.6-3.2.7); LSTM dilatih global (subbab 3.2.7.3c).",
        "Recursive forecasting merekomputasi fitur lag_7/14/21 dan rolling_*_7/14/28 secara dinamis "
        "memakai riwayat aktual+prediksi untuk mencegah leakage nilai aktual dari dalam horizon 28 hari.",
        "Direct forecasting melatih satu model per horizon (1..28) memakai fitur pada hari cutoff (subbab 3.2.8.2).",
        "Evaluasi pada notebook ini memakai main split (d_1-d_1913 vs d_1914-d_1941), bukan 4-fold CV; "
        "4-fold time-series CV, combined_score, best model selection (Top-8/Top-5/Top-10), dan WRMSSE "
        "hierarkis penuh (12 level) didelegasikan ke notebook evaluasi berikutnya (Tahap 8-9).",
        "WRMSSE_proxy pada manifest adalah metrik level-kelompok (bobot dinormalisasi di dalam kelompok), "
        "BUKAN WRMSSE resmi kompetisi M5 yang membutuhkan agregasi ke 12 level hierarki dan bobot dari "
        "seluruh dataset - itu dihitung pada notebook evaluasi setelah semua prediksi digabungkan.",
        "LSTM_MAX_SERIES membatasi jumlah deret waktu untuk sekuens LSTM sebagai adaptasi terhadap "
        "keterbatasan hardware (10 GB RAM usable, 4 GB VRAM); lihat sel Konfigurasi untuk detail dan cara "
        "menaikkannya pada hardware yang lebih besar.",
    ],
    "artifacts": {
        "manifest": str(MANIFEST_PATH.resolve()),
        "model_dir": str(MODEL_DIR.resolve()),
        "prediction_dir": str(PRED_DIR.resolve()),
        "summary": str(SUMMARY_PATH.resolve()),
    },
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

summary

{'source': 'Outline Skripsi Bab 3 - Tahap 6/7: Strategi Grouped Modeling & Model Training (Base Models)',
 'mode': 'full',
 'grouping_strategies': {'byStore': ['store_id'],
  'byStoreCategory': ['store_id', 'cat_id'],
  'byStoreDepartment': ['store_id', 'dept_id']},
 'forecasting_strategies': ['recursive', 'direct'],
 'base_models': ['lightgbm', 'xgboost', 'catboost', 'lstm'],
 'stores_used': ['CA_1',
  'CA_2',
  'CA_3',
  'CA_4',
  'TX_1',
  'TX_2',
  'TX_3',
  'WI_1',
  'WI_2',
  'WI_3'],
 'expected_tree_config_rows': 18,
 'expected_lstm_config_rows': 2,
 'n_manifest_rows': 662,
 'hyperparameters': {'lightgbm': {'objective': 'tweedie',
   'tweedie_variance_power': 1.1,
   'learning_rate': 0.03,
   'n_estimators': 1500,
   'num_leaves': 2047,
   'min_data_in_leaf': 4095,
   'feature_fraction': 0.5,
   'bagging_fraction': 0.5,
   'bagging_freq': 1,
   'random_state': 42,
   'n_jobs': -1,
   'verbose': -1},
  'xgboost': {'objective': 'reg:tweedie',
   'tweedie_variance_power': 1.1,
   '

## 16. Sanity Check

In [ ]:
results_df = load_manifest()

assert not results_df.empty, "Manifest kosong - tidak ada model yang berhasil dilatih."
assert results_df[["RMSE", "MAE"]].notna().all().all(), "Ditemukan NaN pada metrik RMSE/MAE."
assert results_df["strategy"].nunique() >= 1
assert "fold" in results_df.columns, "Kolom fold hilang dari manifest."
expected_fold_labels = {fs["fold"] for fs in FOLD_SPECS} | {"MAIN"}
_unknown_folds = set(results_df["fold"].unique()) - expected_fold_labels
assert not _unknown_folds, f"Ada label fold tak dikenal: {_unknown_folds}"
assert set(results_df["forecasting"].unique()).issubset(set(FORECASTING_STRATEGIES))

n_tree_rows = results_df[results_df["model"].isin(TREE_BASE_MODELS)].shape[0]
n_lstm_rows = results_df[results_df["model"] == "lstm"].shape[0]

print("Baris manifest untuk tree-based models:", n_tree_rows)
print("Baris manifest untuk LSTM (global):", n_lstm_rows)
print("\nRingkasan per model x forecasting (rata-rata WRMSSE_proxy, RMSE):")
display(
    results_df.groupby(["model", "forecasting"])[["WRMSSE_proxy", "RMSE", "MAE", "MAPE"]]
    .mean()
    .sort_values("WRMSSE_proxy")
)

print("\nSemua sanity check lolos. Artefak siap dipakai pada Tahap 8 (Model Evaluation) dan "
      "Tahap 10 (Ensemble Learning: Optimized Weighted Ensemble & Stacking Ensemble Ridge).")

Baris manifest untuk tree-based models: 660
Baris manifest untuk LSTM (global): 2

Ringkasan per model x forecasting (rata-rata WRMSSE_proxy, RMSE):


WRMSSE_proxy      RMSE       MAE       MAPE
model    forecasting                                             
lstm     recursive   -18399.409256  2.299767  1.179915  60.741787
         direct      -16885.698278  2.244975  1.065871  66.204522
lightgbm recursive        0.971091  1.804240  0.973800  59.081323
         direct           0.977004  1.812854  0.959511  58.853790
xgboost  recursive        0.980477  1.831347  0.957241  62.816771
         direct           0.982897  1.834146  0.960389  58.664415
catboost recursive        1.212887  2.538657  1.140492  66.689362
         direct           1.224549  2.618958  1.184362  59.998274


Semua sanity check lolos. Artefak siap dipakai pada Tahap 8 (Model Evaluation) dan Tahap 10 (Ensemble Learning: Optimized Weighted Ensemble & Stacking Ensemble Ridge).
